In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SOURCE_TABLES = {
    # silver source tables
    "brnd": MTU.get_fully_qualified_table_name("oracle", "upc", "pps_pkg_brand"),
    "upc_brnd": MTU.get_fully_qualified_table_name("oracle", "upc", "brand"),
    "prod": MTU.get_fully_qualified_table_name("oracle", "upc", "pps_prod"),
    "home_brwr": MTU.get_fully_qualified_table_name("oracle", "upc", "pps_home_brwr"),
    "edw_extr": MTU.get_fully_qualified_table_name("external", "stlpr205", "edw_prd_mts_extr"),
}

In [0]:
# business transformations here
high_lvl_brnd_famly_nm = (
    F.when(F.col("brnd.brand_prod_typ_cd") == "10", F.lit("AB BRANDS"))
    .when(F.col("brnd.brand_prod_typ_cd") == "20", F.lit("ALLIANCE BRANDS"))
    .when(F.col("brnd.brand_prod_typ_cd") == "30", F.lit("IMPORT BRANDS"))
    .when(F.col("brnd.brand_prod_typ_cd") == "40", F.lit("NON-MALT BRANDS"))
)

In [0]:
# mapping from silver tables to gold brand table
brnd_col_mappings = {
    "brnd_cd": F.col("brnd.pkg_brand_cd"),
    "brnd_nm": F.trim(F.regexp_replace(F.col("brnd.standard_nm"), " +", " ")),
    "brnd_short_nm": F.trim(F.regexp_replace(F.col("brnd.short_nm"), " +", " ")),
    "trdmk_nm": F.trim(F.regexp_replace(F.col("brnd.trademark_nm"), " +", " ")),
    "brnd_typ_cd": F.trim(F.col("brnd.type_cd")),
    "brnd_abbr": F.col("brnd.pkg_brand_cd"),
    "brnd_imp_flg": F.col("brnd.import_flg"),
    "brnd_aseptic_flg": F.col("brnd.aseptic_pkg_flg"),
    "brnd_intrl_nbr": F.col("brnd.internal_nbr"),
    "brnd_cmptv_nbr": F.col("brnd.competive_nbr"),
    "brnd_cmptv_nm": F.trim(F.regexp_replace(F.col("edw_extr.cmptv_brand_nm"), " +", " ")),
    "brewed_prod_cd": F.col("brnd.brewed_product_id"),
    "actv_flg": F.when(F.col("brnd.status_cd") == "A", F.lit("Y")).otherwise(F.lit("N")),
    "high_lvl_brnd_famly_cd": F.col("brnd.brand_prod_typ_cd"),
    "fed_prod_clsfn_cd": F.col("brnd.fpc_tax_cd"),
    "high_lvl_brnd_famly_nm": high_lvl_brnd_famly_nm,
    "mktng_brnd_cd": F.coalesce(F.col("upc_brnd.brand_market_brand_code"), F.col("brnd.pkg_brand_cd")),
    "expanded_alchl_strength_cd": F.col("prod.expanded_alchl_strength_cd"),
    "home_brwr_cd": F.col("home_brwr.home_brwr_code"),
    "home_brwr_nm": F.col("home_brwr.home_brwr_nm"),
    "liquid_typ_cd": F.col("brnd.fpc_tax_cd"),
}

In [0]:
# other tables to join to base
upc_brnd_df = spark.read.table(SOURCE_TABLES["upc_brnd"]).select(
    "brand_code", "brand_market_brand_code", "brand_version_no"
)
upc_brnd_df = (
    upc_brnd_df.groupBy("brand_code")
    .agg(F.max("brand_version_no").alias("brand_version_no"))
    .join(upc_brnd_df, on=["brand_code", "brand_version_no"], how="inner")
    .select("brand_code", "brand_market_brand_code", "brand_version_no")
)

In [0]:
try:
    # Base DF for joins
    left_df = spark.read.table(SOURCE_TABLES["brnd"]).alias("brnd")

    # other tables to join to base
    upc_brnd_df = spark.read.table(SOURCE_TABLES["upc_brnd"]).select(
        "brand_code", "brand_market_brand_code", "brand_version_no"
    )
    # get max brand version and filter upc_brand dataframe to get latest brand codes
    upc_brnd_df = (
        upc_brnd_df.groupBy("brand_code")
        .agg(F.max("brand_version_no").alias("brand_version_no"))
        .join(upc_brnd_df, on=["brand_code", "brand_version_no"], how="inner")
        .select("brand_code", "brand_market_brand_code", "brand_version_no")
        .alias("upc_brnd")
    )
    prod_df = spark.read.table(SOURCE_TABLES["prod"]).select("product_id", "expanded_alchl_strength_cd").alias("prod")
    home_brwr_df = (
        spark.read.table(SOURCE_TABLES["home_brwr"])
        .select(F.col("home_brwr_cd").alias("home_brwr_code"), "home_brwr_nm")
        .alias("home_brwr")
    )
    edw_extr_df = (
        spark.read.table(SOURCE_TABLES["edw_extr"]).select("pkg_brnd_cmptv_nbr", "cmptv_brand_nm").alias("edw_extr")
    )
    # join silver tables
    left_df = left_df.join(
        other=upc_brnd_df, on=F.col("brnd.pkg_brand_cd") == F.col("upc_brnd.brand_code"), how="left_outer"
    )
    left_df = left_df.join(
        other=prod_df, on=F.col("brnd.brewed_product_id") == F.col("prod.product_id"), how="left_outer"
    )
    left_df = left_df.join(
        other=home_brwr_df, on=F.col("brnd.home_brwr_cd") == F.col("home_brwr.home_brwr_code"), how="left_outer"
    )
    left_df = left_df.join(
        other=edw_extr_df, on=F.col("brnd.competive_nbr") == F.col("edw_extr.pkg_brnd_cmptv_nbr"), how="left_outer"
    )

    # Transform and select final columns
    medallion.df = (
        left_df.withColumns(brnd_col_mappings)
        .select(*brnd_col_mappings.keys())
        .withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(TimestampType()))
    )

except Exception as e:
    medallion.logger.error(f"Error processing brand source table reads/joins. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
except Exception as e:
    medallion.logger.error(f"Error writing brand table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )